# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Adtividad en Equipos Semanas 7 y 8 : LDA y LMM audio-a-texto**

* **Nombres y matrículas:**

  *   Alonso Pedrero Martínez | A01769076
  *   A01797612 | Lilian Margarita Álvarez Iglesias
  *   A01235844 | Luis Felipe Neri Alvarado Fregoso

* **Número de Equipo: 14**


* ##### **En cada ejercicio pueden importar los paquetes o librerías que requieran.**

* ##### **En cada ejercicio pueden incluir las celdas y líneas de código que deseen.**

In [9]:
import os
import urllib.request
from pathlib import Path
from pydub import AudioSegment
import speech_recognition as sr
from IPython.display import Markdown, display
import re
import unicodedata
import nltk
from nltk.corpus import stopwords
from unidecode import unidecode
from gensim import corpora
from gensim.models import LdaModel
import ollama
import json
import time


# **Ejercicio 1:**

* #### **Liga de los audios de las fábulas de Esopo:** https://www.gutenberg.org/ebooks/21144

* #### **Descargar los 10 archivos de audio solicitados: 1, 4, 5, 6, 14, 22, 24, 25, 26, 27.**



In [10]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...
DOWNLOAD_LINK = "https://www.gutenberg.org/files/21144/mp3/21144-"
audio_numbers = ["01", "04", "05", "06", "14", "22", "24", "25", "26", "27"]

output_dir_mp3 = "../files/audios_mp3"
os.makedirs(output_dir_mp3, exist_ok=True)

output_dir_wav = "../files/audios_wav"
os.makedirs(output_dir_wav, exist_ok=True)

In [11]:
for num in audio_numbers:
    print(f"Downloading audio {num}...")
    
    url = f"{DOWNLOAD_LINK}{num}.mp3"
    output_path = os.path.join(output_dir_mp3, f"21144-{num}.mp3")
    
    urllib.request.urlretrieve(url, output_path)
    
    print(f"Saved to {output_path}")

Saved to ../files/audios_mp3/21144-01.mp3
Saved to ../files/audios_mp3/21144-04.mp3
Saved to ../files/audios_mp3/21144-05.mp3
Saved to ../files/audios_mp3/21144-06.mp3
Saved to ../files/audios_mp3/21144-14.mp3
Saved to ../files/audios_mp3/21144-22.mp3
Saved to ../files/audios_mp3/21144-24.mp3
Saved to ../files/audios_mp3/21144-25.mp3
Saved to ../files/audios_mp3/21144-26.mp3
Saved to ../files/audios_mp3/21144-27.mp3


In [12]:
output_dir_mp3 = Path("../files/audios_mp3")
output_dir_wav = Path("../files/audios_wav")

output_dir_wav.mkdir(parents=True, exist_ok=True)

for mp3_file in output_dir_mp3.glob("*.mp3"):
    wav_file = output_dir_wav / f"{mp3_file.stem}.wav"

    print(f"Converting {mp3_file.name} -> {wav_file.name}")

    audio = AudioSegment.from_mp3(mp3_file)
    audio.export(wav_file, format="wav")

print("Conversion to wav completed.")

Converting 21144-06.mp3 -> 21144-06.wav
Converting 21144-05.mp3 -> 21144-05.wav
Converting 21144-04.mp3 -> 21144-04.wav
Converting 21144-14.mp3 -> 21144-14.wav
Converting 21144-01.mp3 -> 21144-01.wav
Converting 21144-27.mp3 -> 21144-27.wav
Converting 21144-26.mp3 -> 21144-26.wav
Converting 21144-24.mp3 -> 21144-24.wav
Converting 21144-25.mp3 -> 21144-25.wav
Converting 21144-22.mp3 -> 21144-22.wav
Conversion to wav completed.


# **Ejercicio 2a:**

* #### **Comenten el por qué del modelo seleccionado para extracción del texto de los audios.**

Se seleccionó la herramienta de reconocimiento de voz basada en Google Speech Recognition debido a su facilidad de implementación, rapidez de ejecución y buenos resultados para la transcripción automática de audios en español. Esta solución permitió procesar de manera eficiente la mayoría de las fábulas sin requerir entrenamiento adicional ni infraestructura especializada. Durante las pruebas se identificó un caso particular en el primer audio, donde la introducción provocaba una transcripción incompleta; sin embargo, este problema se resolvió mediante un preprocesamiento sencillo que eliminó los primeros segundos de la grabación. Considerando la relación entre precisión, simplicidad y tiempo de desarrollo, esta herramienta resultó adecuada para los objetivos de la actividad.



* #### **Extraer el contenido de los audios en texto.**

* #### **Sugerencia:** pueden extraerlo en un formato de diccionario, clave:valor $→$ {audio01:fabula01, ...}

In [13]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

def speech_to_text(wav_file_input):
    #Recognizer initiation
    recognizer = sr.Recognizer()

    with sr.AudioFile(str(wav_file_input)) as source:

        audio_data = recognizer.record(source)  # Extract audio info

        try:
            text = recognizer.recognize_google(audio_data,
                                                language="es-MX"
                                                )
            return text

        except sr.UnknownValueError:
            print("Audio failed")
        except sr.RequestError as e:
            print(f"Request failed; {e}")

In [14]:
output_text = {}
count = 0

for wav_file in sorted(output_dir_wav.glob("*.wav")):
    if count == 0:
        audio = AudioSegment.from_wav(str(wav_file))
        audio = audio[15000:]

        temp_file = "21144-01.wav"
        audio.export("../files/audios_wav/" + temp_file, format="wav")

    if wav_file not in output_text:
        output_text[audio_numbers[count]] = speech_to_text(wav_file)
        display(output_text[audio_numbers[count]])

    count += 1


'número 61 El lobo y el cordero en el templo dándose cuenta de que era perseguido por un lobo un pequeño corderito decidió refugiarse en un templo cercano lo llamó el lobo y le dijo que si el sacrificador lo encontraba allí adentro lo emular a su Dios mejor así replicó el cordero Prefiero ser víctima para un Dios a tener que perecer en tus colmillos sí sin remedio vamos a ser sacrificados más nos vale que sea con el mayor honor fin de la fábula Esta es una grabación del dominio público'

'las fábulas de Esopo grabado para librivox.org por Roberto Antonio Muñoz fábula número 64 El lobo y la grulla a un lobo que comía un hueso se le atragantó el hueso en la garganta y corría por todas partes en busca de auxilio encontró en su carrera una grulla y le pidió que le salvará de aquella situación y que enseguida le pagaría por ello aceptó la grulla e introdujo su cabeza en la boca del lobo sacando de la garganta el hueso atravesado pidió entonces la cancelación de la paga convenida Oye amiga dijo el lobo No crees que es suficiente paga con haber sacado tu cabeza sana y salva de mi boca nunca hagas favores a malvados traficantes o corruptos pues mucha paga tendrías si te dejan sano y salvo fin de fábula esta grabación es de dominio público'

'las fábulas de Esopo grabado para librivox.org por Karen savage fábula número 65 El lobo y el caballo pasaba un lobo por un sembrado de cebada pero como no era comida de su gusto la dejó y siguió su camino encontró al rato a un caballo y le llevó al campo comentándole la gran cantidad de cebada que había hallado pero que en vez de comérsela a él mejor se la había dejado porque le agradaba más oír el ruido de sus dientes al masticarla pero el caballo le repuso amigo si los lobos comieran cebada No hubieras preferido complacer a tus oídos sino a tu estómago a todo malvado Aunque parezca actuar como bueno no debe de creérselo fin de fábula esta grabación es de dominio público'

'las fábulas de Esopo grabado para librivox.org por Alejandro González Calderón fábula número 66 El lobo y el asno un lobo fue elegido Rey entre sus congéneres y decretó una ley ordenando que lo que cada uno capturas en la casa lo pusieran en común y lo repartiese por partes iguales entre todos de esta manera ya no tendrían Los Lobos que devorarse unos a otros en épocas de hambre pero en eso le escuchó un asno que estaba por ahí cerca y moviendo sus orejas le dijo magnífica idea ha brotado de tu corazón Pero por qué así escondido todo tu botín en tu cueva llevando a la comunidad y reparte lo también como lo ha decretado el lobo descubierto y confundido derogó su ley Si alguna vez llegas a tener poder de legislar sea el primero en cumplir tus propias leyes fin de la fábula está grabación es del dominio público'

'las fábulas de Esopo grabado para librivox.org por el osito fábula número 74 El lobo y el Cabrito encerrado por la seguridad del corral de una casa un cabrito vio pasar a un lobo y comenzó a insultarle burlándose ampliamente de él el lobo serenamente le replicó infeliz sé que no eres tú quien me está insultando sino el sitio en que te encuentras muy a menudo no es el valor sino la ocasión y el lugar quienes proveen el enfrentamiento arrogante ante los poderosos fin de la fábula está grabación es del dominio público'

'las fábulas de Esopo grabado para librivox.org por el osito fábula número 82 el perro y la almeja un perro de esos acostumbrados a comer huevos al ver una almeja no lo pensó dos veces y creyendo que se trataba de un huevo se la tragó inmediatamente desgarradas luego sus entrañas se sintió muy mal y se dijo bien merecido lo tengo por creer que todo lo que veo redondo son huevos nunca tomes un asunto sin antes reflexionar para no entrar luego en extrañas dificultades fin de la fábula está grabación es del dominio público'

'las fábulas de Esopo grabado para librivox.org por Karen savage fábula número 84 El perro y el reflejo en el río va un perro un río llevando en su hocico un sabroso pedazo de carne vio su propio reflejo en el agua del río y creyó que aquel reflejo era en realidad otro perro que llevaba un trozo de carne mayor que el suyo y deseando adueñarse del pedazo ajeno soltó el suyo para arrebatar el trozo a su supuesto compadre pero el resultado fue que se quedó sin el propio y sin el ajeno este porque no existía solo era un reflejo y el otro el verdadero porque se lo llevó a la corriente nunca codicies el bien ajeno pues puedes perder lo que ya Has adquirido con tu esfuerzo fin de fábula esta grabación es de dominio público'

'las fábulas de Esopo gravado para limpiar box.org fábula número 85 El perro y el carnicero penetró un perro en una carnicería y notando que el carnicero estaba muy ocupado con sus clientes cogió un trozo de carne y salió corriendo se volvió el carnicero y viéndole huir y sin poder hacer nada exclamó Oye amigo ahí donde te encuentre no dejaré de mirarte no esperes a que suceda un accidente para pensar en cómo evitarlo fin de fábula está agrupación es de dominio público'

'las fábulas de Esopo grabado para librivox.org por el osito fábula número 86 el perro con campanilla había un perro que acostumbraba a morder sin razón le puso su amo una campanilla para advertirle a la gente de su presencia cercana y el can sonando la campanilla se fue a la plaza pública a presumir más una savia perra ya avanzada de años le dijo de qué presumes tanto amigo sé que no llevas esa campanilla por tus grandes virtudes sino para anunciar tu maldad oculta los halagos que se hacen a sí mismo los fanfarrones solo delatan sus mayores defectos fin de la fábula está grabación es del dominio público'

'las fábulas de Esopo gravado para librivox.org por el osito fábula número 87 el perro que perseguía al león un perro de casa se encontró con un león y partió en su persecución pero el león se volvió rugiendo y el perro todo atemorizado retrocedió rápidamente por el mismo camino le dio una zorra y le dijo perro infeliz primero perseguía al león y ya ni siquiera soporta surgidos cuando entres a una empresa mantente siempre listo a afrontar imprevistos que no te imaginabas fin de la fábula está grabación es del dominio público'

# **Ejercicio 2b:**

* #### **Eliminar el inicio y final comunes de los textos extraídos de cada fábula.**

* #### **Sugerencia:** Pueden guardar esta información en un archivo tipo JSON, para que al estar probando diferentes opciones en los ejercicios siguientes, puedan recuperar rápidamente la información de cada video/fábula.

In [15]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...
for value in output_text.values():
    print(value)
    print("-" * 50)

número 61 El lobo y el cordero en el templo dándose cuenta de que era perseguido por un lobo un pequeño corderito decidió refugiarse en un templo cercano lo llamó el lobo y le dijo que si el sacrificador lo encontraba allí adentro lo emular a su Dios mejor así replicó el cordero Prefiero ser víctima para un Dios a tener que perecer en tus colmillos sí sin remedio vamos a ser sacrificados más nos vale que sea con el mayor honor fin de la fábula Esta es una grabación del dominio público
--------------------------------------------------
las fábulas de Esopo grabado para librivox.org por Roberto Antonio Muñoz fábula número 64 El lobo y la grulla a un lobo que comía un hueso se le atragantó el hueso en la garganta y corría por todas partes en busca de auxilio encontró en su carrera una grulla y le pidió que le salvará de aquella situación y que enseguida le pagaría por ello aceptó la grulla e introdujo su cabeza en la boca del lobo sacando de la garganta el hueso atravesado pidió entonces 

In [16]:
for key, value in output_text.items():
    if value:
        value = re.sub(
            r"las\s+f[áa]bulas?\s+de\s+esopo.*?f[áa]bula\s+n[uú]mero\s+\d+",
            "",
            value,
            flags=re.IGNORECASE | re.DOTALL
        )

        value = re.sub(
            r"fin\s+de\s+f[áa]bula.*?dominio\s+p[úu]blico",
            "",
            value,
            flags=re.IGNORECASE | re.DOTALL
        )

        value = re.sub(
            r"fin\s+de la\s+f[áa]bula.*?dominio\s+p[úu]blico",
            "",
            value,
            flags=re.IGNORECASE | re.DOTALL
        )

        output_text[key] = value.strip()

In [17]:
for value in output_text.values():
    print(value)
    print("-" * 50)

número 61 El lobo y el cordero en el templo dándose cuenta de que era perseguido por un lobo un pequeño corderito decidió refugiarse en un templo cercano lo llamó el lobo y le dijo que si el sacrificador lo encontraba allí adentro lo emular a su Dios mejor así replicó el cordero Prefiero ser víctima para un Dios a tener que perecer en tus colmillos sí sin remedio vamos a ser sacrificados más nos vale que sea con el mayor honor
--------------------------------------------------
El lobo y la grulla a un lobo que comía un hueso se le atragantó el hueso en la garganta y corría por todas partes en busca de auxilio encontró en su carrera una grulla y le pidió que le salvará de aquella situación y que enseguida le pagaría por ello aceptó la grulla e introdujo su cabeza en la boca del lobo sacando de la garganta el hueso atravesado pidió entonces la cancelación de la paga convenida Oye amiga dijo el lobo No crees que es suficiente paga con haber sacado tu cabeza sana y salva de mi boca nunca h

# **Ejercicio 3:**

* #### **Apliquen el proceso de limpieza que consideren adecuado.**

* #### **Justifiquen los pasos de limpieza utilizados. Tomen en cuenta que el texto extraído de cada fábula es relativamente pequeño.**

* #### **En caso de que decidan no aplicar esta etapa de limpieza, deberán justificarlo.**

Se realizó una etapa de limpieza y normalización de texto con el objetivo de garantizar la consistencia de los datos antes de aplicar técnicas de análisis de lenguaje natural. Aunque la transcripción fue obtenida directamente en español y presentó una calidad adecuada, se decidió convertir todo el texto a minúsculas, eliminar acentos, signos de puntuación y espacios redundantes para reducir posibles variaciones en la representación de las palabras. Este preprocesamiento no era estrictamente necesario para comprender el contenido de las fábulas, pero se implementó como una medida preventiva para evitar inconsistencias durante etapas posteriores como la extracción de palabras clave, el análisis de frecuencias y el modelado de temas mediante LDA, mejorando así la uniformidad y robustez del conjunto de datos.


In [18]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

for key, value in output_text.items():
    if value:
        value = value.lower()

        value = ''.join(
            c for c in unicodedata.normalize('NFD', value)
            if unicodedata.category(c) != 'Mn'
        )

        value = re.sub(r"[^\w\s]", "", value)

        value = re.sub(r"\s+", " ", value)

        output_text[key] = value.strip()



In [19]:
for value in output_text.values():
    print(value)
    print("-" * 50)

numero 61 el lobo y el cordero en el templo dandose cuenta de que era perseguido por un lobo un pequeno corderito decidio refugiarse en un templo cercano lo llamo el lobo y le dijo que si el sacrificador lo encontraba alli adentro lo emular a su dios mejor asi replico el cordero prefiero ser victima para un dios a tener que perecer en tus colmillos si sin remedio vamos a ser sacrificados mas nos vale que sea con el mayor honor
--------------------------------------------------
el lobo y la grulla a un lobo que comia un hueso se le atraganto el hueso en la garganta y corria por todas partes en busca de auxilio encontro en su carrera una grulla y le pidio que le salvara de aquella situacion y que enseguida le pagaria por ello acepto la grulla e introdujo su cabeza en la boca del lobo sacando de la garganta el hueso atravesado pidio entonces la cancelacion de la paga convenida oye amiga dijo el lobo no crees que es suficiente paga con haber sacado tu cabeza sana y salva de mi boca nunca h

# **Ejercicio 4:**

In [20]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

nltk.download("stopwords")

stop_words = set(stopwords.words("spanish"))

def tokeinze_text(text):
    tokens = [
        word
        for word in text.split()
        if word not in stop_words and len(word) > 2
    ]

    return tokens


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/alonsopedreromartinez/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [21]:
tokenized_output_text = {}

for item, value in output_text.items():
    tokenized_output_text[item] = tokeinze_text(value)

In [22]:
tokenized_output_text.items()

dict_items([('01', ['numero', 'lobo', 'cordero', 'templo', 'dandose', 'cuenta', 'perseguido', 'lobo', 'pequeno', 'corderito', 'decidio', 'refugiarse', 'templo', 'cercano', 'llamo', 'lobo', 'dijo', 'sacrificador', 'encontraba', 'alli', 'adentro', 'emular', 'dios', 'mejor', 'asi', 'replico', 'cordero', 'prefiero', 'ser', 'victima', 'dios', 'tener', 'perecer', 'colmillos', 'remedio', 'vamos', 'ser', 'sacrificados', 'mas', 'vale', 'mayor', 'honor']), ('04', ['lobo', 'grulla', 'lobo', 'comia', 'hueso', 'atraganto', 'hueso', 'garganta', 'corria', 'todas', 'partes', 'busca', 'auxilio', 'encontro', 'carrera', 'grulla', 'pidio', 'salvara', 'aquella', 'situacion', 'enseguida', 'pagaria', 'ello', 'acepto', 'grulla', 'introdujo', 'cabeza', 'boca', 'lobo', 'sacando', 'garganta', 'hueso', 'atravesado', 'pidio', 'entonces', 'cancelacion', 'paga', 'convenida', 'oye', 'amiga', 'dijo', 'lobo', 'crees', 'suficiente', 'paga', 'haber', 'sacado', 'cabeza', 'sana', 'salva', 'boca', 'nunca', 'hagas', 'favores

In [23]:
def extract_keywords_lda(tokens, num_topics=20, num_words=10):

    documents = [tokens]

    dictionary = corpora.Dictionary(documents)
    corpus = [dictionary.doc2bow(doc) for doc in documents]

    lda = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=min(num_topics, len(dictionary)),
        passes=20,
        random_state=42
    )

    principal_topic = lda.show_topic(0, topn=num_words)

    return [word for word, weight in principal_topic]

In [24]:
lda_output_text = {}

for key, item in tokenized_output_text.items():
    lda_output_text[key] = extract_keywords_lda(item)

print(lda_output_text.items())

dict_items([('01', ['replico', 'sacrificador', 'pequeno', 'perecer', 'perseguido', 'prefiero', 'refugiarse', 'remedio', 'mejor', 'templo']), ('04', ['oye', 'mucha', 'partes', 'pagaria', 'paga', 'pues', 'nunca', 'introdujo', 'lobo', 'sacado']), ('05', ['oidos', 'parezca', 'hallado', 'llevo', 'lobo', 'lobos', 'malvado', 'mas', 'masticarla', 'mejor']), ('06', ['lobo', 'llegas', 'manera', 'magnifica', 'lobos', 'ordenando', 'llevando', 'legislar', 'ley', 'partes']), ('14', ['replico', 'seguridad', 'ocasion', 'pasar', 'poderosos', 'proveen', 'lugar', 'valor', 'sino', 'lobo']), ('22', ['reflexionar', 'sintio', 'merecido', 'nunca', 'penso', 'perro', 'redondo', 'luego', 'trataba', 'veces']), ('24', ['reflejo', 'resultado', 'perder', 'perro', 'propio', 'puedes', 'pues', 'quedo', 'realidad', 'nunca']), ('25', ['pensar', 'poder', 'notando', 'ocupado', 'oye', 'penetro', 'huir', 'perro', 'viendole', 'suceda']), ('26', ['presumes', 'presumir', 'mismo', 'morder', 'oculta', 'perra', 'perro', 'plaza', '

# **Ejercicio 5a y 5b:**

* #### **5a: Mediante el LLM que hayan seleccionado, generar un único enunciado que describa o resuma cada fábula.**

* #### **5b: Mediante el LLM que hayan seleccionado, generar tres posibles enunciados diferentes relacionados con la historia de la fábula.**

* #### **Sugerencia:** En realidad los dos incisos a y b se pueden obtener con un solo prompt que solicite la información y el formato correspondiente para cada una de estas partes. Por ejemplo, para cada fábula la salida puede ser un primer enunciado genérico que resume o describe dicha temática; seguido de tres enunciados, cada uno hablando sobre una situación o parte diferente de la fábula.

In [28]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...
for key, value in lda_output_text.items():

    prompt = f"""
    Analiza la siguiente texto de la palabras de una fábula extraidas por Latent Dirichchlet Allocation.

    FÁBULA:
    {value}

    Devuelve únicamente  las repuestas en el formato JSON siguiente,
    no me regreses nada mas que el resumen y los 3 enunciados de posibles subtemas,
    tampoco me digas que si, solo dame las respuestas en el formato,
    No digas entendido o mas cosas, solo el resumen y los 3 eneunciados en el siguiente formato estilo JSON:

    {{
      "resumen": "",
      "enunciados": [
        "",
        "",
        ""
      ]
    }}
    """

    answer = ollama.chat(
        model="llama3.2",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    print(key)
    print(answer["message"]["content"])


01
Dentro de la fábula extraída por Latent Dirichlet Allocation, las palabras 'replico', 'sacrificador', 'pequeno', 'perecer', 'perseguido', 'prefiero', 'refugiarse', 'remedio', 'mejor' y 'templo' pueden ser asociadas a los siguientes enunciados y resumen:

{
  "resumen": "La fábula describe un personaje que debe enfrentar sus propias debilidades y encontrar una solución para superarlos.",
  "enunciados": [
    "El personaje se ve obligado a replicar ciertas características negativas para sobrevivir en su entorno.",
    "Pero al hacerlo, se da cuenta de que es un sacrificio que le impide ser completamente libre.",
    "En busca de una salida mejor, el personaje debe encontrar un remedio para curarse a sí mismo y escapar de sus problemas."
  ]
}
04
Bueno, no tengo información sobre una fábula extraída por Latent Dirichlet Allocation. Sin embargo, puedo intentar analizar las palabras y generar un resumen basado en la información proporcionada.

Aquí te dejo el resultado:

```
{
  "resume

# **Ejercicio 6:**

* #### **Incluyan sus conclusiones de la actividad audio-a-texto:**


En conclusión, la actividad permitió construir un flujo completo de procesamiento de lenguaje natural que abarca desde la descarga automática de los audios, su conversión y transcripción a texto, hasta la extracción de temas mediante modelado LDA. Aunque el modelo fue alimentado con cada fábula de manera individual, lo que puede limitar parcialmente la calidad y diversidad de los temas generados debido al tamaño reducido de cada documento, los resultados obtenidos fueron consistentes y permitieron identificar correctamente los conceptos principales presentes en las narraciones. LDA demostró ser una técnica clásica, accesible y fácil de interpretar dentro del campo del NLP, ya que permite descubrir temas latentes sin necesidad de etiquetas o conocimiento previo sobre el contenido. Además, la automatización de todas las etapas del proceso convirtió el ejercicio en un ejemplo práctico de un pipeline de análisis de datos de extremo a extremo, donde la información fluye automáticamente desde la fuente original hasta la obtención de resultados analíticos útiles.


# **Fin de la actividad LDA y LMM: audio-a-texto**